# 13. Ejemplo de embeddings preentrenados

En el notebook `05_BoW_Tfidf.ipynb` vimos que Bag of Words y TF-IDF representan cada documento como un vector de frecuencias sobre el vocabulario del corpus. Estas representaciones son **sparse** (la mayoría de posiciones son cero), tienen tantas dimensiones como palabras del vocabulario, y no capturan ninguna relación de significado entre palabras: para BoW, "perro" y "gato" son tan distintas entre sí como "perro" y "coche".

Los **embeddings** resuelven esta limitación: representan cada palabra como un vector **denso** de dimensión fija (por ejemplo, 50 o 128), aprendido a partir de grandes cantidades de texto de forma que palabras con significados o contextos de uso parecidos obtienen vectores próximos entre sí.

En la siguiente imagen podemos ver esta diferencia entre un **vector sparse** y un **embedding** o vector denso.

<img src="imgs/011_embeddings_vs_sparse.png" style="width: 900px;"/>

En este notebook usamos un modelo de embeddings **ya entrenado**, publicado en TensorFlow Hub, en lugar de entrenar uno desde cero. Esto tiene una ventaja práctica clara: aprovechamos el conocimiento aprendido a partir de un corpus enorme sin necesitar ni ese corpus ni la capacidad de cómputo para entrenarlo.

El modelo que usamos es **NNLM** (*Neural Network Language Model*) en español:

- `nnlm-es-dim50`: embeddings de 50 dimensiones.
- `nnlm-es-dim128`: embeddings de 128 dimensiones (más expresivos, pero más pesados).

Basta cambiar la URL del modelo para elegir una versión u otra.

> **Nota**: la primera ejecución descarga el modelo desde TensorFlow Hub y requiere conexión a Internet. Las siguientes ejecuciones reutilizan una copia cacheada localmente.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # silencia los mensajes informativos de TensorFlow

import tensorflow as tf
tf.get_logger().setLevel('ERROR')          # silencia los WARNING de la API antigua de TF

import tensorflow_hub as hub

---

## 1. Carga del modelo y vectorización de palabras

Cargamos el modelo NNLM de 50 dimensiones y lo aplicamos directamente sobre una lista de palabras. El modelo se comporta como una función: se le pasa una lista de textos y devuelve un array de NumPy con un vector por texto.

In [2]:
# Se puede elegir la version de 50 o de 128 dimensiones cambiando la URL
# La primera ejecucion descarga el modelo y requiere conexion a Internet
modelo = hub.load('https://tfhub.dev/google/nnlm-es-dim50/2')   # 50 dimensiones

palabras = ['perro', 'gato', 'coche']
vectores = modelo(palabras).numpy()

print('Dimension de cada vector:', vectores.shape[1])   # -> 50
for palabra, v in zip(palabras, vectores):
    print(f"'{palabra}': {v.round(4)} ...")

Dimension de cada vector: 50
'perro': [ 0.077   0.3244  0.1317  0.112  -0.1765  0.1041  0.0171  0.1795  0.2507
 -0.0577  0.1967 -0.1559 -0.239  -0.1517  0.082   0.2057  0.1415  0.0378
 -0.0712  0.0579  0.0439  0.1041 -0.1222  0.022  -0.1113 -0.0795 -0.0035
 -0.027  -0.0123 -0.1359  0.091   0.1033 -0.117  -0.0359  0.2131 -0.202
  0.1063 -0.2189  0.2163  0.1722  0.1793  0.0496  0.0211 -0.0071  0.0729
  0.0546 -0.0922 -0.1284 -0.1379 -0.2946] ...
'gato': [ 8.500e-02  3.553e-01  4.990e-02  1.471e-01  1.046e-01  1.615e-01
 -1.199e-01  2.276e-01  9.110e-02 -1.029e-01  3.823e-01 -9.800e-03
 -1.098e-01 -4.310e-02  1.377e-01  2.753e-01  7.300e-03  1.548e-01
  2.000e-04 -4.270e-02  5.420e-02 -1.220e-02 -1.725e-01  5.040e-02
 -1.067e-01 -1.850e-01 -1.192e-01  2.840e-02  3.390e-02 -2.119e-01
  4.770e-02  1.223e-01 -1.220e-01 -1.900e-01  2.676e-01 -1.152e-01
  1.384e-01 -1.256e-01  1.794e-01  3.360e-02  2.620e-02  1.990e-02
 -3.700e-02  9.690e-02  1.032e-01  2.950e-02 -1.760e-01  4.930e-02
 -5.670e

como vemos cada palabra queda representada por un vector de 50 números reales (solo mostramos 4 decimales)

A simple vista estos números no dicen mucho: la información semántica de un embedding no está en ninguna dimensión concreta, sino en la posición relativa del vector respecto a los vectores de otras palabras. Para comprobar que el modelo captura significado, en la siguiente sección medimos la **similitud** entre palabras.

---

## 2. Similitud semántica entre palabras

Si los embeddings capturan significado, palabras relacionadas deberían tener vectores más parecidos entre sí que palabras no relacionadas. Lo comprobamos con seis palabras agrupadas en tres parejas: dos animales, dos vehículos y dos términos de realeza.

Usamos la **similitud coseno** entre vectores, que mide el ángulo entre ellos: cuanto más próxima a 1, más parecida es la dirección de los vectores (y, por tanto, más relacionado está su significado); cuanto más próxima a 0 o negativa, menos relacionados están.

In [3]:
import numpy as np
import pandas as pd

palabras = ['perro', 'gato', 'coche', 'moto', 'python', 'java']
vectores = modelo(palabras).numpy()

def similitud_coseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

n = len(palabras)
matriz_sim = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        matriz_sim[i, j] = similitud_coseno(vectores[i], vectores[j])

pd.DataFrame(matriz_sim.round(3), index=palabras, columns=palabras)

,perro,gato,coche,moto,python,java
perro,1.000,0.744,0.427,0.415,0.261,0.236
gato,0.744,1.000,0.270,0.247,0.409,0.233
coche,0.427,0.270,1.000,0.839,0.105,0.246
moto,0.415,0.247,0.839,1.000,0.126,0.199
python,0.261,0.409,0.105,0.126,1.000,0.770
java,0.236,0.233,0.246,0.199,0.770,1.000


## Conclusiones

Los resultados confirman que el modelo captura relaciones semánticas reales, sin que nadie le haya indicado explícitamente qué palabras están relacionadas:

- **"perro" y "gato"** obtienen una similitud alta (en torno a 0.74): ambas son mascotas domésticas.
- **"coche" y "moto"** obtienen una similitud aún más alta (en torno a 0.84): ambos son vehículos.
- **"python" y "java"** también muestran una similitud clara (en torno a 0.77).
- Las combinaciones entre categorías distintas, como "perro" y "coche" o "gato" y "moto", obtienen similitudes notablemente más bajas.

Esto es precisamente lo que no podíamos obtener con Bag of Words o TF-IDF: en esas representaciones, dos palabras solo se consideran "parecidas" si aparecen literalmente en los mismos documentos. Con embeddings preentrenados, la similitud de significado está codificada directamente en el vector, aprendida de antemano a partir de un corpus mucho más grande que el nuestro.

Esta propiedad es la que hace que los embeddings sean el punto de partida habitual de las redes neuronales modernas para NLP: en lugar de alimentar el modelo con vectores sparse de frecuencias, se le alimenta con vectores densos que ya incorporan una noción de significado.

---

## Bonus Track

TensorFlow Hub es un repositorio de modelos de aprendizaje automático ya entrenados, junto con una librería cliente en Python (`tensorflow_hub`) para descargarlos y reutilizarlos directamente en un proyecto, sin necesidad de entrenarlos ni de definir su arquitectura desde cero.

Su funcionamiento se resume en dos ideas:

* Repositorio (`tfhub.dev`): alberga modelos preentrenados de distintos dominios —embeddings de texto, clasificación de imágenes, detección de objetos, etc.— publicados por Google y por la comunidad.
* Librería cliente (`tensorflow_hub`): con una función tan simple como hub.load(url), descarga el modelo indicado, lo cachea localmente y lo deja listo para usarse como una función invocable dentro de TensorFlow.

En este notebook se usa precisamente así: hub.load('https://tfhub.dev/google/nnlm-es-dim50/2') descarga un modelo de embeddings de palabras en español ya entrenado por Google, y a partir de ahí basta llamarlo con una lista de palabras para obtener sus vectores, sin haber entrenado nada.